# Framingham Heart Study — CHD Risk Prediction Pipeline
### Complete End-to-End Decision Tree Classification Pipeline

**Dataset:** Framingham Heart Study · 4 240 rows · 15 features · Binary target: TenYearCHD

**Pipeline Steps:**
1. Data Loading & EDA
2. Preprocessing — NaN Imputation
3. Feature Selection (Filter · Wrapper · Embedded)
4. Apply Selected Features
5. Baseline Model
6. Hyperparameter Tuning (GridSearchCV · RandomizedSearchCV · Bayesian Optuna)
7. Cross Validation (KFold · StratifiedKFold · LOOCV · LeavePOut)
8. Bias-Variance Tradeoff (Learning Curves · Decomposition)
9. Final Model Comparison

# STEP 1 — Data Loading & EDA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv(r"C:\Users\vishwa\Downloads\archive (18)\framingham.csv")

In [ ]:
print(f"Shape : {df.shape}")
print(f"Columns : {list(df.columns)}")
df.head()

In [ ]:
df.info()

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal NaN : {df.isnull().sum().sum()}")

In [ ]:
df.describe().round(2)

In [ ]:
print("Target distribution:")
print(df['TenYearCHD'].value_counts())
print(f"\nPositive rate : {df['TenYearCHD'].mean()*100:.1f}%  (class imbalance present)")

In [ ]:
# Skewness check — justifies median imputation over mean
median_cols = ['glucose', 'totChol', 'BMI', 'heartRate', 'cigsPerDay']
for col in median_cols:
    print(f"{col:<15}: skewness = {df[col].skew():.2f}")

### Note — No Feature Engineering for Numerical Columns
Decision Tree splits on **value thresholds**, not distances.
- No StandardScaler required for numerical columns
- OHE / OrdinalEncoder not required (all features already numerical)
- Only NaN imputation is needed before feature selection methods

# STEP 2 — Feature / Target Split & Train-Test Split

In [ ]:
X = df[['male', 'age', 'education', 'currentSmoker', 'cigsPerDay', 'BPMeds',
       'prevalentStroke', 'prevalentHyp', 'diabetes', 'totChol', 'sysBP',
       'diaBP', 'BMI', 'heartRate', 'glucose']]

y = df[['TenYearCHD']]

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    train_size   = 0.8,
    random_state = 42,
    stratify     = y          # preserves 15% CHD ratio in both splits
)

print(f"X_train : {X_train.shape}  |  CHD rate: {y_train.values.mean()*100:.1f}%")
print(f"X_test  : {X_test.shape}  |  CHD rate: {y_test.values.mean()*100:.1f}%")

## NaN Imputation
> Decision Tree handles NaN natively — imputation here is ONLY for feature selection methods that cannot handle NaN.
> `X_train_imp` / `X_test_imp` are used for feature selection and bias-variance only.

In [ ]:
from sklearn.impute import SimpleImputer

median_cols = ['glucose', 'totChol', 'BMI', 'heartRate', 'cigsPerDay']
mode_cols   = ['BPMeds', 'education']

imp_median = SimpleImputer(strategy='median')
imp_mode   = SimpleImputer(strategy='most_frequent')

X_train_imp = X_train.copy()
X_test_imp  = X_test.copy()

# fit only on train — apply to both
X_train_imp[median_cols] = imp_median.fit_transform(X_train[median_cols])
X_test_imp[median_cols]  = imp_median.transform(X_test[median_cols])

X_train_imp[mode_cols]   = imp_mode.fit_transform(X_train[mode_cols])
X_test_imp[mode_cols]    = imp_mode.transform(X_test[mode_cols])

print(f"NaN remaining — X_train_imp : {X_train_imp.isnull().sum().sum()}")
print(f"NaN remaining — X_test_imp  : {X_test_imp.isnull().sum().sum()}")

# STEP 3 — Feature Selection

Three categories covering all methods from the reference notebook:

| Category | Method | Key |
|---|---|---|
| Filter | VarianceThreshold | Remove near-zero variance |
| Filter | SelectKBest (f_classif) | ANOVA F-test ranking |
| Filter | mutual_info_classif | Information-theoretic ranking |
| Wrapper | RFE | Recursive model-based elimination |
| Embedded | feature_importances_ | Built-in DT importance score |

## 3.1 Filter — VarianceThreshold (Zero / Low Variance)

In [ ]:
from sklearn.feature_selection import VarianceThreshold

vt = VarianceThreshold(threshold=0.1)
vt.fit(X_train_imp)

kept    = list(vt.get_feature_names_out())
removed = [c for c in X_train_imp.columns if c not in kept]

print(f"Features kept    ({len(kept)})  : {kept}")
print(f"Features removed ({len(removed)}): {removed}")

## 3.2 Filter — SelectKBest (ANOVA F-test)

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

kbest = SelectKBest(f_classif, k=10)
kbest.fit(X_train_imp, y_train.values.ravel())

kbest_features = list(kbest.get_feature_names_out())

scores_df = pd.DataFrame({
    'Feature': X_train_imp.columns,
    'F_Score': kbest.scores_
}).sort_values('F_Score', ascending=False)

plt.figure(figsize=(10, 4))
sns.barplot(data=scores_df, x='Feature', y='F_Score', palette='Blues_r')
plt.xticks(rotation=45, ha='right')
plt.title('SelectKBest — ANOVA F-Scores (higher = more relevant to target)')
plt.tight_layout()
plt.show()

print("Top 10 selected by SelectKBest:", kbest_features)

## 3.3 Filter — Mutual Information

In [ ]:
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(X_train_imp, y_train.values.ravel(), random_state=42)

mi_df = pd.DataFrame({
    'Feature' : X_train_imp.columns,
    'MI_Score': mi_scores
}).sort_values('MI_Score', ascending=False)

plt.figure(figsize=(10, 4))
sns.barplot(data=mi_df, x='Feature', y='MI_Score', palette='Greens_r')
plt.xticks(rotation=45, ha='right')
plt.title('Mutual Information Scores (higher = stronger dependency with target)')
plt.tight_layout()
plt.show()

mi_features = mi_df.head(10)['Feature'].tolist()
print("Top 10 by Mutual Information:", mi_features)

## 3.4 Wrapper — RFE (Recursive Feature Elimination)

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeClassifier

rfe = RFE(
    estimator            = DecisionTreeClassifier(random_state=42),
    n_features_to_select = 10
)
rfe.fit(X_train_imp, y_train.values.ravel())

rfe_features = X_train_imp.columns[rfe.support_].tolist()

rfe_rank_df = pd.DataFrame({
    'Feature': X_train_imp.columns,
    'Rank'   : rfe.ranking_       # rank 1 = selected
}).sort_values('Rank')

print("Features selected by RFE (rank=1):", rfe_features)
print()
print(rfe_rank_df.to_string(index=False))

## 3.5 Embedded — Decision Tree Feature Importances

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Fit a quick DT on imputed data to get feature importances
dt_for_fi = DecisionTreeClassifier(random_state=42)
dt_for_fi.fit(X_train_imp, y_train.values.ravel())

fi_df = pd.DataFrame({
    'Feature'   : X_train_imp.columns,
    'Importance': dt_for_fi.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 4))
sns.barplot(data=fi_df, x='Feature', y='Importance', palette='Oranges_r')
plt.xticks(rotation=45, ha='right')
plt.title('Decision Tree Feature Importances (Embedded Method — Gini)')
plt.tight_layout()
plt.show()

fi_features = fi_df.head(10)['Feature'].tolist()
print("Top 10 by Feature Importance:", fi_features)

## 3.6 Feature Selection Summary — Consensus Table

In [ ]:
from collections import Counter

# Features selected by each method (top 10 each)
method_results = {
    'SelectKBest' : set(kbest_features),
    'Mutual Info' : set(mi_features),
    'RFE'         : set(rfe_features),
    'DT Importnc' : set(fi_features)
}

all_feats = list(X_train_imp.columns)
summary_data = []

for feat in all_feats:
    row = {'Feature': feat}
    count = 0
    for method, selected in method_results.items():
        picked = '✓' if feat in selected else '✗'
        row[method] = picked
        if feat in selected:
            count += 1
    row['Votes'] = count
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data).sort_values('Votes', ascending=False)
print(summary_df.to_string(index=False))

# STEP 4 — Apply Selected Features

Using **RFE-selected features** as the training feature set.
RFE directly evaluates model performance to choose features — most appropriate for a DT pipeline.

In [ ]:
# Final selected feature set from RFE
selected_features = rfe_features
print(f"Selected {len(selected_features)} features: {selected_features}")

# Apply to imputed train/test sets
X_train_sel = X_train_imp[selected_features]
X_test_sel  = X_test_imp[selected_features]

print(f"\nX_train_sel : {X_train_sel.shape}")
print(f"X_test_sel  : {X_test_sel.shape}")

# STEP 5 — Baseline Model (Before Tuning)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report)

DT1 = DecisionTreeClassifier(random_state=42)
DT1.fit(X_train_sel, y_train.values.ravel())

y_pred1 = DT1.predict(X_test_sel)
y_prob1 = DT1.predict_proba(X_test_sel)[:, 1]

print('=' * 45)
print('  BASELINE — DecisionTreeClassifier')
print('=' * 45)
print(f"  Accuracy  : {accuracy_score(y_test, y_pred1):.4f}")
print(f"  Precision : {precision_score(y_test, y_pred1):.4f}")
print(f"  Recall    : {recall_score(y_test, y_pred1):.4f}")
print(f"  F1 Score  : {f1_score(y_test, y_pred1):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(y_test, y_prob1):.4f}")
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred1))
print()
print("Classification Report:")
print(classification_report(y_test, y_pred1))

# STEP 6 — Hyperparameter Tuning

## 6.1 GridSearchCV — Exhaustive Search

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth'        : [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
    'criterion'        : ['gini', 'entropy']
}

grid_search = GridSearchCV(
    estimator  = DecisionTreeClassifier(random_state=42),
    param_grid = param_grid,
    cv         = 5,
    scoring    = 'roc_auc',
    n_jobs     = -1,
    verbose    = 1
)

grid_search.fit(X_train_sel, y_train.values.ravel())

print("Best Parameters :", grid_search.best_params_)
print("Best ROC-AUC    :", round(grid_search.best_score_, 4))

DT_grid = grid_search.best_estimator_

## 6.2 RandomizedSearchCV — Random Sampling

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'max_depth'        : [3, 5, 7, 10, 15, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf' : [1, 2, 4, 8],
    'criterion'        : ['gini', 'entropy'],
    'max_features'     : ['sqrt', 'log2', None]
}

random_search = RandomizedSearchCV(
    estimator           = DecisionTreeClassifier(random_state=42),
    param_distributions = param_dist,
    n_iter              = 40,
    cv                  = 5,
    scoring             = 'roc_auc',
    random_state        = 42,
    n_jobs              = -1,
    verbose             = 1
)

random_search.fit(X_train_sel, y_train.values.ravel())

print("Best Parameters :", random_search.best_params_)
print("Best ROC-AUC    :", round(random_search.best_score_, 4))

DT_random = random_search.best_estimator_

## 6.3 Bayesian Optimization — Optuna

In [ ]:
import optuna
from sklearn.model_selection import KFold, cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Create Objective Function
def Objective(trial):
    # create search space
    max_depth         = trial.suggest_int('max_depth', 2, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf  = trial.suggest_int('min_samples_leaf', 1, 10)
    criterion         = trial.suggest_categorical('criterion', ['gini', 'entropy'])

    # train the algorithm
    dt = DecisionTreeClassifier(
        max_depth         = max_depth,
        min_samples_split = min_samples_split,
        min_samples_leaf  = min_samples_leaf,
        criterion         = criterion,
        random_state      = 42
    )

    # get the Performance metrics
    kf    = KFold(n_splits=5, shuffle=True, random_state=23)
    score = cross_val_score(estimator=dt, X=X_train_sel,
                            y=y_train.values.ravel(),
                            scoring='roc_auc', cv=kf).mean()
    return score

# 2. Create the Study
study = optuna.create_study(direction='maximize')

# 3. Evaluate model Performance
study.optimize(Objective, n_trials=40, show_progress_bar=True)

In [ ]:
print("Best Parameters :", study.best_params)
print("Best ROC-AUC    :", round(study.best_value, 4))

DT_bayesian = DecisionTreeClassifier(**study.best_params, random_state=42)
DT_bayesian.fit(X_train_sel, y_train.values.ravel())

# STEP 7 — Cross Validation

Applied on **DT_bayesian** (best tuned model) using **X_train_sel** (selected features).
Scoring: `roc_auc` — most appropriate for imbalanced CHD data (15% positive).

| Strategy | Folds | Best For |
|---|---|---|
| KFold-5 | 5 | General purpose |
| KFold-10 | 10 | Lower bias estimate |
| StratifiedKFold-5 | 5 | **Imbalanced classes — recommended here** |
| LOOCV | n | Small datasets (slow) |
| LeavePOut | C(n,p) | Educational |

## 7.1 K-Fold (5 splits)

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

k5     = KFold(n_splits=5, shuffle=True, random_state=44)
sc_k5  = cross_val_score(DT_bayesian, X_train_sel, y_train.values.ravel(),
                          cv=k5, scoring='roc_auc')

print("KFold-5  per fold :", sc_k5.round(4))
print(f"Mean : {sc_k5.mean():.4f}  |  Std : {sc_k5.std():.4f}")

## 7.2 K-Fold (10 splits)

In [ ]:
k10    = KFold(n_splits=10, shuffle=True, random_state=44)
sc_k10 = cross_val_score(DT_bayesian, X_train_sel, y_train.values.ravel(),
                          cv=k10, scoring='roc_auc')

print("KFold-10 per fold :", sc_k10.round(4))
print(f"Mean : {sc_k10.mean():.4f}  |  Std : {sc_k10.std():.4f}")

## 7.3 Stratified K-Fold (5 splits) — Recommended

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=44)
sc_skf = cross_val_score(DT_bayesian, X_train_sel, y_train.values.ravel(),
                          cv=skf, scoring='roc_auc')

print("StratifiedKFold-5 per fold :", sc_skf.round(4))
print(f"Mean : {sc_skf.mean():.4f}  |  Std : {sc_skf.std():.4f}")

## 7.4 Leave One Out CV (LOOCV)

In [ ]:
from sklearn.model_selection import LeaveOneOut

loocv    = LeaveOneOut()
sc_loocv = cross_val_score(DT_bayesian, X_train_sel, y_train.values.ravel(),
                             cv=loocv, scoring='accuracy', n_jobs=-1)

print(f"LOOCV  Mean Accuracy : {sc_loocv.mean():.4f}  |  Std : {sc_loocv.std():.4f}")

## 7.5 Leave P Out CV (p=2)

In [ ]:
from sklearn.model_selection import LeavePOut

lpov    = LeavePOut(p=2)
sc_lpov = cross_val_score(DT_bayesian, X_train_sel, y_train.values.ravel(),
                           cv=lpov, scoring='accuracy', n_jobs=-1)

print(f"LeavePOut(p=2) Mean Accuracy : {sc_lpov.mean():.4f}  |  Std : {sc_lpov.std():.4f}")

## 7.6 Cross Validation Summary

In [ ]:
cv_summary = pd.DataFrame({
    'Strategy'  : ['KFold-5',          'KFold-10',         'StratifiedKFold-5', 'LOOCV',           'LeavePOut(p=2)'],
    'Metric'    : ['ROC-AUC',          'ROC-AUC',          'ROC-AUC',           'Accuracy',        'Accuracy'],
    'Mean Score': [sc_k5.mean(),       sc_k10.mean(),      sc_skf.mean(),       sc_loocv.mean(),   sc_lpov.mean()],
    'Std Score' : [sc_k5.std(),        sc_k10.std(),       sc_skf.std(),        sc_loocv.std(),    sc_lpov.std()]
})

cv_summary['Mean Score'] = cv_summary['Mean Score'].round(4)
cv_summary['Std Score']  = cv_summary['Std Score'].round(4)

print(cv_summary.to_string(index=False))

# STEP 8 — Bias-Variance Tradeoff

Three approaches:
1. **sklearn `learning_curve`** — manual plot of train vs validation score with std bands
2. **mlxtend `bias_variance_decomp`** — numerical decomposition: Expected Loss = Bias² + Variance
3. **mlxtend `plot_learning_curves`** — quick visual (reference notebook style)

> Uses `X_train_sel` (imputed + feature-selected, no NaN)

## 8.1 Learning Curves — sklearn

In [ ]:
from sklearn.model_selection import learning_curve

dt_lc = DecisionTreeClassifier(**study.best_params, random_state=42)

train_sizes, train_scores, val_scores = learning_curve(
    estimator    = dt_lc,
    X            = X_train_sel,
    y            = y_train.values.ravel(),
    cv           = 5,
    scoring      = 'roc_auc',
    train_sizes  = np.linspace(0.1, 1.0, 10),
    n_jobs       = -1,
    random_state = 42
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, 'o-', color='royalblue',  label='Training Score')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                 alpha=0.15, color='royalblue')
plt.plot(train_sizes, val_mean,   'o-', color='seagreen',   label='Validation Score')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                 alpha=0.15, color='seagreen')
plt.xlabel('Training Samples')
plt.ylabel('ROC-AUC Score')
plt.title('Learning Curves — Bias-Variance Tradeoff\n(DT Bayesian Best Model on Selected Features)')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8.2 Bias-Variance Decomposition — mlxtend

In [ ]:
from mlxtend.evaluate import bias_variance_decomp

dt_bv = DecisionTreeClassifier(**study.best_params, random_state=42)

avg_expected_loss, avg_bias, avg_var = bias_variance_decomp(
    dt_bv,
    X_train_sel.values,
    y_train.values.ravel(),
    X_test_sel.values,
    y_test.values.ravel(),
    loss        = '0-1_loss',
    random_seed = 42
)

print('=' * 45)
print('  Bias-Variance Decomposition — DT Bayesian')
print('=' * 45)
print(f'  Average Expected Loss : {avg_expected_loss:.4f}')
print(f'  Average Bias²         : {avg_bias:.4f}')
print(f'  Average Variance      : {avg_var:.4f}')
print('-' * 45)
print(f'  Bias² + Variance      : {avg_bias + avg_var:.4f}  (should ≈ Expected Loss)')
print('=' * 45)

## 8.3 Learning Curves — mlxtend

In [ ]:
from mlxtend.plotting import plot_learning_curves

dt_lc2 = DecisionTreeClassifier(**study.best_params, random_state=42)

plot_learning_curves(
    X_train_sel.values,
    y_train.values.ravel(),
    X_test_sel.values,
    y_test.values.ravel(),
    clf = dt_lc2
)
plt.title('Learning Curves — DT Bayesian Best Model (mlxtend)')
plt.tight_layout()
plt.show()

# STEP 9 — Final Model Comparison

All four models evaluated on the **held-out test set** using selected features.

Metrics:
- **Accuracy** — overall correctness
- **Recall** — sensitivity (critical: missing a CHD case is costly)
- **F1 Score** — harmonic mean of precision and recall
- **ROC-AUC** — overall discrimination ability (primary metric for imbalanced data)

In [ ]:
from sklearn.metrics import (accuracy_score, recall_score,
                              f1_score, roc_auc_score,
                              confusion_matrix, classification_report)

all_models = {
    'DT Baseline'    : DT1,
    'DT GridSearch'  : DT_grid,
    'DT RandomSearch': DT_random,
    'DT Bayesian'    : DT_bayesian
}

results = []

print(f"{'Model':<20} {'Accuracy':>10} {'Recall':>10} {'F1':>10} {'ROC-AUC':>10}")
print('-' * 62)

for name, model in all_models.items():
    y_pred = model.predict(X_test_sel)
    y_prob = model.predict_proba(X_test_sel)[:, 1]

    acc  = accuracy_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    roc  = roc_auc_score(y_test, y_prob)

    results.append({'Model': name, 'Accuracy': acc,
                    'Recall': rec, 'F1': f1, 'ROC-AUC': roc})

    print(f"{name:<20} {acc:>10.4f} {rec:>10.4f} {f1:>10.4f} {roc:>10.4f}")

In [ ]:
# Best model by ROC-AUC
results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)

print("\n=== Best model by ROC-AUC ===")
best_row = results_df.iloc[0]
print(f"Model     : {best_row['Model']}")
print(f"ROC-AUC   : {best_row['ROC-AUC']:.4f}")
print(f"Recall    : {best_row['Recall']:.4f}")
print(f"F1 Score  : {best_row['F1']:.4f}")
print(f"Accuracy  : {best_row['Accuracy']:.4f}")

In [ ]:
# Confusion matrix for best model (DT_bayesian)
best_pred = DT_bayesian.predict(X_test_sel)
cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No CHD', 'CHD'],
            yticklabels=['No CHD', 'CHD'])
plt.title('Confusion Matrix — DT Bayesian (Best Model)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

print("\nClassification Report — DT Bayesian:")
print(classification_report(y_test, best_pred, target_names=['No CHD', 'CHD']))

In [ ]:
# ROC Curve comparison for all models
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(9, 7))

for name, model in all_models.items():
    y_prob = model.predict_proba(X_test_sel)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance of the final best model (DT_bayesian)
fi_final = pd.DataFrame({
    'Feature'   : selected_features,
    'Importance': DT_bayesian.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(9, 4))
sns.barplot(data=fi_final, x='Feature', y='Importance', palette='Reds_r')
plt.xticks(rotation=45, ha='right')
plt.title('Feature Importances — DT Bayesian Final Model')
plt.tight_layout()
plt.show()

print(fi_final.to_string(index=False))